In [ ]:
import pickle
from tqdm import tqdm
import numpy as np
# import gymnasium_env
from gymnasium_env.envs.blokus_piece import BlokusPiece, BlokusPieceTransformations
PIECE_IDS = ['1', '2', 'I3', 'V3', 'I4', 'L4', 'O', 'T4', 'Z4', 'F', 'I5', 'L5', 'N', 'P', 'T5', 'U', 'V5', 'W', 'X', 'Y', 'Z5']

In [ ]:
def encode(x, y, idx):
    x *= -1
    y *= -1
    return (x << 13) + (y << 10) + idx

def decode(action):
    x = (action >> 13) * -1
    y = ((action >> 10) & 7) * -1
    idx = action & 1023
    return x, y, idx

def compute_neighborhood_positions():
    cnt = 0    
    neighbor_idx, neighbor_pos = {}, []
    for i in range(-2, 1):
        for j in range(1, 4 - (-i // 2)):
            neighbor_idx[i, j] = cnt
            neighbor_pos.append((i, j))
            print(i, j, cnt)
            cnt += 1
    for i in range(1, 3):
        for j in range(-2, 5 - i):
            neighbor_idx[i, j] = cnt
            neighbor_pos.append((i, j))
            print(i, j, cnt)
            cnt += 1
    for i in range(-1, 2):
        neighbor_idx[3, i] = cnt
        neighbor_pos.append((3, i))
        print(-3, i, cnt)
        cnt += 1
    directory = 'pre_neighbors'
    with open(f'{directory}/neighbor_idx.pkl', 'wb') as f:
        pickle.dump(neighbor_idx, f)
    with open(f'{directory}/neighbor_pos.pkl', 'wb') as f:
        pickle.dump(neighbor_pos, f)
    return neighbor_idx, neighbor_pos

def compute_neighborhood_actions():
    to_idx, to_pos = compute_neighborhood_positions()
    pieces = {id: BlokusPieceTransformations(id=id) for id in PIECE_IDS}
    _get_piece_info = []
    compute_actions = [[] for _ in range(1 << 22)]
    for i in range(len(pieces)):
        for j in range(len(pieces[PIECE_IDS[i]].transformations)):
            pieces[PIECE_IDS[i]].transformations[j].idx = len(_get_piece_info)
            _get_piece_info.append((PIECE_IDS[i], j))

    assert len(_get_piece_info) == 91
    # print(to_idx)
    for i in tqdm(range(91)):
        id, idx = _get_piece_info[i]
        for h, w in pieces[id].transformations[idx].shape:
            # print(h, w, "Hey1")
            good_cover = True
            ids = []
            for xx, yy in pieces[id].transformations[idx].shape:
                x, y = xx - h, yy - w
                # print(x, y)
                if not ((x, y) in to_idx) and not (x == 0 and y == 0):
                    good_cover = False
                    break
                elif x != 0 or y != 0:
                    ids.append(to_idx[(x, y)])
                pass
            # print(ids)
            if not good_cover:
                continue
            # print(ids)
            for j in range(1 << 22):
                all_ids = True
                # print(ids)
                for k in ids:
                    if not (j & (1 << k)):
                        all_ids = False
                if all_ids:
                    # print("YEsS", h, w, i)
                    compute_actions[j].append(encode(-h, -w, i))
    sum = 0
    # Convert compute_actions to a numpy array with dtype int32
    compute_actions = np.array([np.array(actions, dtype=np.int16) for actions in compute_actions], dtype=object)

    # Save the numpy array to a pickle file
    # directory = 'pre_neighbors'
    # with open(f'{directory}/compute_actions.pkl', 'wb') as f:
    #     pickle.dump(compute_actions, f)

    # # Calculate the sum
    for i in range(1 << 22):
        sum += len(compute_actions[i])
    print(sum)
    print(sum)
    return compute_actions



In [ ]:
positions = compute_neighborhood_positions()

In [ ]:
compute_actions = compute_neighborhood_actions()
import pickle
with open('pre_neighbors/compute_actions.pkl', 'rb') as f:
    compute_actions = pickle.load(f)
    
with open('pre_neighbors/neighbor_idx.pkl', 'rb') as f:
    neighbor_idx = pickle.load(f)

with open('pre_neighbors/neighbor_pos.pkl', 'rb') as f:
    neighbor_pos = pickle.load(f)

# print(compute_actions)
# print(neighbor_idx)
# print(neighbor_pos)

In [ ]:
print(51380224>>22)

In [ ]:
print(len(compute_actions[1 << 22 - 1]))
print(compute_actions[1 << 22 - 1])
# print(list(map(decode, compute_actions[(1 << 22) - 1])))